# 🩸 Anemo AI — Training Pipeline (Google Colab)

**One-click GPU training for anemia detection models.**

This notebook:
1. Verifies GPU availability
2. Installs all dependencies
3. Configures Kaggle credentials
4. Downloads anemia datasets from Kaggle
5. Organises dataset into train/val/test splits
6. Trains EfficientNetV2-S models for 3 body parts
7. Converts trained models to TF.js INT8-quantized format
8. Downloads models as a ZIP for deployment

**Before running:**
- Set runtime to GPU: Runtime → Change runtime type → T4 GPU
- Run cells in order (Shift+Enter)

**Expected training time:** ~2-4 hours on T4 GPU

In [1]:
# ── Cell 1: GPU Check ─────────────────────────────────────────────────────
import subprocess
import sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('⚠ No GPU detected!')
    print('Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU')

import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
for gpu in gpus:
    details = tf.config.experimental.get_device_details(gpu)
    print(f'  → {details.get("device_name", gpu)}')
    tf.config.experimental.set_memory_growth(gpu, True)

if not gpus:
    print('\n⚠ WARNING: No GPU found. Training will be very slow on CPU.')
    print('  → Enable GPU in Runtime settings and restart notebook.')

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [ ]:
# ── Cell 2: Install Dependencies ──────────────────────────────────────────
import subprocess

packages = [
    'kaggle>=1.6.0',
    'albumentations>=1.3.0',
    'tensorflowjs>=4.10.0',
    'scikit-learn>=1.3.0',
    'opencv-python-headless',
    'tqdm',
]

print('Installing dependencies...')
result = subprocess.run(
    ['pip', 'install', '-q'] + packages,
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✓ All dependencies installed')
else:
    print('⚠ Some packages may have issues:')
    print(result.stderr[-500:] if result.stderr else '(no output)')

# Verify critical imports
import_checks = ['kaggle', 'albumentations', 'cv2', 'sklearn']
for pkg in import_checks:
    try:
        __import__(pkg)
        print(f'  ✓ {pkg}')
    except ImportError as e:
        print(f'  ✗ {pkg}: {e}')

In [ ]:
# ── Cell 3: Kaggle Credentials ────────────────────────────────────────────
import json
import os
from pathlib import Path

# ─── SET YOUR CREDENTIALS ─────────────────────────────────────────────────
KAGGLE_USERNAME = 'cyrilcquinoviva'
KAGGLE_KEY      = 'KGAT_4c4efbb044d9da5e17af934f94de5acd'
# ──────────────────────────────────────────────────────────────────────────

# Write kaggle.json
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'
kaggle_json.write_text(json.dumps({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}))
kaggle_json.chmod(0o600)

# Also set environment variables
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

# Verify
import kaggle
try:
    kaggle.api.authenticate()
    print(f'✓ Kaggle authenticated as: {KAGGLE_USERNAME}')
except Exception as e:
    print(f'✗ Kaggle auth failed: {e}')
    print('  Check your username and API key at kaggle.com → Account')

In [ ]:
# ── Cell 4: Setup Workspace ───────────────────────────────────────────────
import os
from pathlib import Path

WORKSPACE = Path('/content/anemo_training')
DATASET_RAW  = WORKSPACE / 'dataset' / 'raw'
DATASET_PROC = WORKSPACE / 'dataset' / 'processed'
MODELS_OUT   = WORKSPACE / 'models_output'
PUBLIC_MODELS = WORKSPACE / 'public' / 'models'

for d in [DATASET_RAW, DATASET_PROC, MODELS_OUT, PUBLIC_MODELS]:
    d.mkdir(parents=True, exist_ok=True)

print(f'✓ Workspace: {WORKSPACE}')
print(f'  dataset/raw:       {DATASET_RAW}')
print(f'  dataset/processed: {DATASET_PROC}')
print(f'  models_output:     {MODELS_OUT}')
print(f'  public/models:     {PUBLIC_MODELS}')

In [ ]:
# ── Cell 5: Download Datasets ─────────────────────────────────────────────
import kaggle
from pathlib import Path

DATASETS = [
    # (dataset_id,                                      dest_subdir,       priority)
    ('amandam1/anemia-dataset',                         'conjunctiva',      1),
    ('longntt2001/anemia-detection-from-nailbeds',      'nailbed',          1),
    ('thefearlesscoder/nail-dataset-for-blood-hemoglobin-estimation', 'nailbed_hgb', 2),
    ('ehababoelnaga/anemia-types-classification',       'clinical',         2),
    ('omkar-thombre/anemia-detection-dataset',          'conjunctiva_palm', 2),
]

downloaded = {}
for dataset_id, dest_name, priority in sorted(DATASETS, key=lambda x: x[2]):
    dest = DATASET_RAW / dest_name

    # Skip if already present
    if dest.exists():
        imgs = list(dest.rglob('*.jpg')) + list(dest.rglob('*.png'))
        if len(imgs) > 10:
            print(f'↩ {dataset_id}: {len(imgs)} images already downloaded')
            downloaded[dest_name] = dest
            continue

    dest.mkdir(parents=True, exist_ok=True)
    print(f'⬇ Downloading {dataset_id}...')
    try:
        kaggle.api.dataset_download_files(dataset_id, path=str(dest), unzip=True, quiet=False)
        imgs = list(dest.rglob('*.jpg')) + list(dest.rglob('*.png'))
        print(f'  ✓ Downloaded: {len(imgs)} images')
        downloaded[dest_name] = dest
    except Exception as e:
        print(f'  ✗ Failed ({priority=}): {e}')
        if priority == 1:
            print('  ⚠ This is a primary dataset — check your credentials and try again.')

print(f'\n✓ Downloaded {len(downloaded)} datasets')

In [ ]:
# ── Cell 6: Organise Dataset ──────────────────────────────────────────────
import random
import shutil
from pathlib import Path

random.seed(42)

CLASS_NAMES = ['0_Normal', '1_Mild', '2_Moderate', '3_Severe']
SPLITS = {'train': 0.70, 'val': 0.15, 'test': 0.15}


def organise_binary_dataset(raw_dir, output_base, body_part,
                            positive_kw=None, negative_kw=None):
    positive_kw = positive_kw or ['anemia', 'anemic', 'positive', '1']
    negative_kw = negative_kw or ['normal', 'healthy', 'negative', '0', 'non']

    raw_dir = Path(raw_dir)
    if not raw_dir.exists():
        print(f'  ⚠ Not found: {raw_dir}')
        return 0

    exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    imgs = [f for ext in exts for f in raw_dir.rglob(f'*{ext}')]
    if not imgs:
        print(f'  ⚠ No images in {raw_dir}')
        return 0

    anemic, normal, unclassified = [], [], []
    for img in imgs:
        path_str = str(img).lower()
        parent_str = img.parent.name.lower()
        check_str = path_str + ' ' + parent_str
        is_pos = any(k in check_str for k in positive_kw)
        is_neg = any(k in check_str for k in negative_kw)
        if is_pos and not is_neg:
            anemic.append(img)
        elif is_neg and not is_pos:
            normal.append(img)
        else:
            unclassified.append(img)

    # Default unclassified to normal (conservative)
    normal.extend(unclassified)

    print(f'  {body_part}: {len(anemic)} anemic (→1_Mild), {len(normal)} normal (→0_Normal)')

    for label, files in [('1_Mild', anemic), ('0_Normal', normal)]:
        random.shuffle(files)
        n = len(files)
        n_train = int(n * 0.70)
        n_val = int(n * 0.15)
        split_data = {
            'train': files[:n_train],
            'val': files[n_train:n_train + n_val],
            'test': files[n_train + n_val:]
        }
        for split_name, split_files in split_data.items():
            dest = Path(output_base) / body_part / split_name / label
            dest.mkdir(parents=True, exist_ok=True)
            for src in split_files:
                try:
                    shutil.copy2(src, dest / src.name)
                except Exception:
                    pass
    return len(imgs)


print('Organising datasets...')
total = 0

# Conjunctiva sources
for conjunctiva_src in ['conjunctiva', 'conjunctiva_palm']:
    n = organise_binary_dataset(DATASET_RAW / conjunctiva_src, DATASET_PROC, 'conjunctiva')
    total += n

# Nail bed sources
for nail_src in ['nailbed', 'nailbed_hgb']:
    n = organise_binary_dataset(DATASET_RAW / nail_src, DATASET_PROC, 'fingernails')
    total += n

print(f'\n✓ Total images organised: {total}')

# Count final distribution
print('\nDataset summary:')
for bp in ['conjunctiva', 'fingernails', 'skin']:
    bp_dir = DATASET_PROC / bp
    if not bp_dir.exists():
        print(f'  {bp}: no data')
        continue
    bp_total = sum(1 for f in bp_dir.rglob('*.jpg')) + sum(1 for f in bp_dir.rglob('*.png'))
    print(f'  {bp}: {bp_total} images')

In [ ]:
# ── Cell 7: Train Models ──────────────────────────────────────────────────
import numpy as np
import cv2
import albumentations as A
import json
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TerminateOnNaN
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# Enable mixed precision on GPU
mixed_precision.set_global_policy('mixed_float16')
print(f'Mixed precision: {mixed_precision.global_policy().name}')

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
CLASS_NAMES = ['0_Normal', '1_Mild', '2_Moderate', '3_Severe']
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


def apply_clahe(img):
    img = np.clip(img, 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge([cl, a, b]), cv2.COLOR_LAB2RGB).astype(np.float32)


def preprocess(img):
    return tf.keras.applications.efficientnet_v2.preprocess_input(apply_clahe(img))


train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.12, rotate_limit=20, p=0.6),
    A.GaussNoise(var_limit=(5.0, 25.0), p=0.3),
    A.CoarseDropout(max_holes=6, max_height=20, max_width=20, p=0.25),
])


def load_split(data_dir, body_part, split, augment=False):
    images, labels = [], []
    split_dir = data_dir / body_part / split
    if not split_dir.exists():
        return np.array([]), np.array([])
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = split_dir / class_name
        if not class_dir.exists():
            continue
        for img_path in list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')):
            try:
                img = img_to_array(load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))).astype(np.uint8)
                if augment:
                    img = train_aug(image=img)['image']
                images.append(preprocess(img))
                labels.append(class_idx)
            except Exception:
                pass
    if not images:
        return np.array([]), np.array([])
    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.int32)
    idx = np.random.permutation(len(X))
    return X[idx], y[idx]


def ordinal_focal_loss(gamma=2.0, label_smoothing=0.05):
    def loss_fn(y_true, y_pred):
        y_smooth = y_true * (1 - label_smoothing) + label_smoothing / NUM_CLASSES
        ce = tf.keras.losses.categorical_crossentropy(y_smooth, y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1)
        fl = tf.pow(1.0 - p_t, gamma) * ce
        true_rank = tf.cast(tf.argmax(y_true, axis=-1), tf.float32)
        pred_rank = tf.cast(tf.argmax(y_pred, axis=-1), tf.float32)
        rank_penalty = tf.abs(true_rank - pred_rank) / float(NUM_CLASSES - 1)
        return fl + 0.25 * rank_penalty
    loss_fn.__name__ = 'ordinal_focal_loss'
    return loss_fn


def build_model():
    base = EfficientNetV2S(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='swish')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='swish')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32', name='predictions')(x)
    return keras.Model(inp, out), base


trained_models = {}
MODELS_OUT.mkdir(exist_ok=True)

for body_part in ['conjunctiva', 'fingernails', 'skin']:
    print(f'\n{"="*50}')
    print(f'TRAINING: {body_part.upper()}')
    print(f'{"="*50}')

    X_train, y_train = load_split(DATASET_PROC, body_part, 'train', augment=True)
    X_val,   y_val   = load_split(DATASET_PROC, body_part, 'val',   augment=False)

    if len(X_train) == 0:
        print(f'  ⚠ No training data — skipping {body_part}')
        continue

    print(f'  Train: {len(X_train)}, Val: {len(X_val)}')

    y_train_ohe = np.eye(NUM_CLASSES)[y_train]
    y_val_ohe   = np.eye(NUM_CLASSES)[y_val] if len(y_val) > 0 else None

    # Class weights for imbalanced data
    classes_present = np.unique(y_train)
    cw = compute_class_weight('balanced', classes=classes_present, y=y_train)
    class_weights = {int(c): float(w) for c, w in zip(classes_present, cw)}
    for i in range(NUM_CLASSES):
        if i not in class_weights:
            class_weights[i] = 1.0
    print(f'  Class weights: {class_weights}')

    model, base = build_model()
    best_path = str(MODELS_OUT / f'anemia_{body_part}_best.h5')
    val_data = (X_val, y_val_ohe) if y_val_ohe is not None else None

    base_callbacks = [
        ModelCheckpoint(best_path, save_best_only=True, monitor='val_accuracy', verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-9, verbose=1),
        TerminateOnNaN(),
    ]

    # ── Phase 1: Head only ──────────────────────────────────────────────────
    print('\n  Phase 1: Training head (frozen base)...')
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3, weight_decay=1e-5),
        loss=ordinal_focal_loss(),
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    model.fit(X_train, y_train_ohe,
              validation_data=val_data,
              epochs=20, batch_size=BATCH_SIZE,
              class_weight=class_weights,
              callbacks=base_callbacks, verbose=1)

    # ── Phase 2: Fine-tune top 30% ──────────────────────────────────────────
    print('\n  Phase 2: Fine-tuning top 30% layers...')
    base.trainable = True
    freeze_until = int(len(base.layers) * 0.7)
    for i, layer in enumerate(base.layers):
        layer.trainable = i >= freeze_until
    unfrozen = sum(1 for l in base.layers if l.trainable)
    print(f'  Unfrozen {unfrozen}/{len(base.layers)} base layers')

    model.compile(
        optimizer=keras.optimizers.Adam(5e-6, weight_decay=1e-5),
        loss=ordinal_focal_loss(),
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    model.fit(X_train, y_train_ohe,
              validation_data=val_data,
              epochs=40, batch_size=BATCH_SIZE,
              class_weight=class_weights,
              callbacks=base_callbacks, verbose=1)

    # Load best & evaluate
    if Path(best_path).exists():
        model.load_weights(best_path)

    X_test, y_test = load_split(DATASET_PROC, body_part, 'test', augment=False)
    if len(X_test) > 0:
        y_test_ohe = np.eye(NUM_CLASSES)[y_test]
        _, test_acc, test_auc = model.evaluate(X_test, y_test_ohe, verbose=0)
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        print(f'\n  Test accuracy: {test_acc:.4f} | AUC: {test_auc:.4f}')
        print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))
    else:
        test_acc = 0.0

    # Save with metadata
    final_path = MODELS_OUT / f'anemia_{body_part}_best.h5'
    model.save(str(final_path))
    (MODELS_OUT / f'anemia_{body_part}_metadata.json').write_text(json.dumps({
        'body_part': body_part,
        'architecture': 'EfficientNetV2-S',
        'input_shape': [IMG_SIZE, IMG_SIZE, 3],
        'num_classes': NUM_CLASSES,
        'class_names': CLASS_NAMES,
        'test_accuracy': float(test_acc),
        'trained_images': len(X_train),
    }, indent=2))

    trained_models[body_part] = str(final_path)
    print(f'  ✓ Saved: {final_path}')

print(f'\n✓ Training complete. Models: {list(trained_models.keys())}')

In [ ]:
# ── Cell 8: Convert to TF.js ──────────────────────────────────────────────
import subprocess
import shutil
from pathlib import Path

BODY_PART_REGISTRY = {
    'conjunctiva': [
        'judges/efficientnet-b0',
        'scouts/squeezenet-1.1-eye',
        'specialists/densenet121',
        'judges/vit-tiny',
        'judges/mlp-meta-learner',
    ],
    'fingernails': [
        'scouts/mobilenet-v3-nails',
        'specialists/inceptionv3',
    ],
    'skin': [
        'scouts/mobilenet-v3-skin',
        'specialists/resnet50v2',
        'specialists/vgg16',
    ],
}

converted = []
for body_part, h5_path in trained_models.items():
    print(f'\nConverting {body_part}...')
    h5_path = Path(h5_path)
    if not h5_path.exists():
        print(f'  ✗ Not found: {h5_path}')
        continue

    temp_dir = MODELS_OUT / f'tfjs_{body_part}'
    temp_dir.mkdir(parents=True, exist_ok=True)

    # Try INT8 quantized first, fallback to no quantization
    for quant_flag in ['--quantize_uint8=*', None]:
        cmd = [
            'tensorflowjs_converter',
            '--input_format=keras',
            '--output_format=tfjs_graph_model',
        ]
        if quant_flag:
            cmd.append(quant_flag)
        cmd += [str(h5_path), str(temp_dir)]

        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if result.returncode == 0 and (temp_dir / 'model.json').exists():
            quantized = 'INT8' if quant_flag else 'none'
            print(f'  ✓ Converted (quantization: {quantized})')
            break
        else:
            if quant_flag:
                print(f'  ⚠ INT8 failed, trying without quantization...')
            else:
                print(f'  ✗ Conversion failed: {result.stderr[-200:]}')
                break

    if not (temp_dir / 'model.json').exists():
        continue

    # Deploy to all registry paths
    registry_paths = BODY_PART_REGISTRY.get(body_part, ['judges/efficientnet-b0'])
    for reg_path in registry_paths:
        dest = PUBLIC_MODELS / reg_path
        dest.mkdir(parents=True, exist_ok=True)
        for f in temp_dir.iterdir():
            if f.is_file():
                shutil.copy2(f, dest / f.name)
        size_kb = sum(f.stat().st_size for f in dest.rglob('*') if f.is_file()) / 1024
        print(f'  ✓ → public/models/{reg_path} ({size_kb:.0f} KB)')

    shutil.rmtree(temp_dir, ignore_errors=True)
    converted.append(body_part)

print(f'\n✓ TF.js conversion done: {converted}')

In [ ]:
# ── Cell 9: Download Models ───────────────────────────────────────────────
import shutil
from google.colab import files
from pathlib import Path

print('Creating deployment ZIP...')
zip_path = '/content/anemo_models_export'

# Include: model.json files + metadata (not .bin shards for size)
export_dir = Path('/content/anemo_models_export_tmp')
export_dir.mkdir(exist_ok=True)

# Copy public/models structure
shutil.copytree(
    str(PUBLIC_MODELS),
    str(export_dir / 'public' / 'models'),
    dirs_exist_ok=True
)

# Also copy the .h5 files
h5_dir = export_dir / 'keras_models'
h5_dir.mkdir(exist_ok=True)
for h5 in MODELS_OUT.glob('*.h5'):
    shutil.copy2(h5, h5_dir / h5.name)

for meta in MODELS_OUT.glob('*_metadata.json'):
    shutil.copy2(meta, h5_dir / meta.name)

# Create zip
shutil.make_archive(zip_path, 'zip', export_dir)
zip_file = zip_path + '.zip'
size_mb = Path(zip_file).stat().st_size / (1024 * 1024)
print(f'\n✓ ZIP created: {zip_file} ({size_mb:.1f} MB)')

# Download to local machine
print('Downloading to your machine...')
files.download(zip_file)
print('\n✓ Download started!')
print('\nAFTER DOWNLOAD:')
print('  1. Extract the ZIP in your project root')
print('  2. Files go to public/models/ (model.json + .bin shards)')
print('  3. Run: npm run dev')
print('  4. Check browser console for: [ConsensusEngine] Model loaded: ...')

In [ ]:
# ── Cell 10: Final Summary ────────────────────────────────────────────────
from pathlib import Path

print('=' * 50)
print('ANEMO AI TRAINING SUMMARY')
print('=' * 50)

print('\nTrained models:')
for h5 in sorted(MODELS_OUT.glob('*_best.h5')):
    meta_path = MODELS_OUT / h5.name.replace('_best.h5', '_metadata.json')
    size_mb = h5.stat().st_size / (1024 * 1024)
    if meta_path.exists():
        import json
        meta = json.loads(meta_path.read_text())
        acc = meta.get('test_accuracy', 0)
        n = meta.get('trained_images', 0)
        print(f'  {h5.name}: acc={acc:.4f} ({acc*100:.1f}%) | trained on {n} images | {size_mb:.1f} MB')
    else:
        print(f'  {h5.name}: {size_mb:.1f} MB')

print('\nDeployed TF.js models:')
total_kb = 0
for model_json in sorted(PUBLIC_MODELS.rglob('model.json')):
    rel = model_json.parent.relative_to(PUBLIC_MODELS)
    size_kb = sum(f.stat().st_size for f in model_json.parent.rglob('*') if f.is_file()) / 1024
    total_kb += size_kb
    print(f'  /models/{rel}  ({size_kb:.0f} KB)')

print(f'\n  Total deployed: {total_kb:.0f} KB ({total_kb/1024:.1f} MB)')
print('\n✓ All done!')